# 🎾 Capítulo 1 — El Fin de la Era de los Big 3
### Análisis del dominio de Federer, Nadal y Djokovic vs la Nueva Generación (2015-2024)
**Fuente de datos:** [Jeff Sackmann — tennis_atp](https://github.com/JeffSackmann/tennis_atp)  
**Torneos analizados:** Grand Slam y Masters 1000 (2015-2024)

---
## 1. Imports y Carga de Datos

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from itertools import product

# Cargar datos 2015-2024
anos = list(range(2015, 2025))
dfs = []

for ano in anos:
    url = f"https://raw.githubusercontent.com/JeffSackmann/tennis_atp/master/atp_matches_{ano}.csv"
    df_ano = pd.read_csv(url)
    df_ano['season'] = ano
    dfs.append(df_ano)
    print(f"✅ {ano} cargado — {df_ano.shape[0]} partidos")

df = pd.concat(dfs, ignore_index=True)
print(f"\nTotal partidos: {df.shape[0]}")
print(f"Total columnas: {df.shape[1]}")

---
## 2. Definición de Grupos y Preparación de Datos

In [ ]:
big3 = ['Roger Federer', 'Rafael Nadal', 'Novak Djokovic']

nueva_gen = ['Carlos Alcaraz', 'Jannik Sinner',
             'Alexander Zverev', 'Daniil Medvedev',
             'Stefanos Tsitsipas', 'Holger Rune']

todos_anos = list(range(2015, 2025))
todos_grupos = ['Big 3', 'Nueva Generacion', 'Resto']

def clasificar_jugador(nombre):
    if nombre in big3:
        return 'Big 3'
    elif nombre in nueva_gen:
        return 'Nueva Generacion'
    else:
        return 'Resto'

# Filtrar solo Slams y Masters 1000
df_grandes = df[df['tourney_level'].isin(['G', 'M'])].copy()

# Solo finales (ganadores de cada torneo)
finales = df_grandes[df_grandes['round'] == 'F'].copy()
finales['grupo_ganador'] = finales['winner_name'].apply(clasificar_jugador)

# Títulos por grupo por año
titulos_por_ano = finales.groupby(
    ['season', 'grupo_ganador']
).size().reset_index(name='titulos')

# Rellenar combinaciones sin títulos con cero
index_completo = pd.DataFrame(
    list(product(todos_anos, todos_grupos)),
    columns=['season', 'grupo_ganador']
)

titulos_completo = index_completo.merge(
    titulos_por_ano, on=['season', 'grupo_ganador'], how='left'
)
titulos_completo['titulos'] = titulos_completo['titulos'].fillna(0)

print(titulos_por_ano)

---
## 3. Visualización 1 — Títulos por Grupo por Año

El gráfico central del proyecto. Muestra el dominio del Big 3 desde 2015 y cómo la Nueva Generación fue escalando hasta tomar el control en 2024.

In [ ]:
colores = {
    'Big 3':            '#1a78cf',
    'Nueva Generacion': '#e8462a',
    'Resto':            '#a0a0a0'
}

fig, ax = plt.subplots(figsize=(14, 7))

for grupo in todos_grupos:
    datos = titulos_completo[titulos_completo['grupo_ganador'] == grupo]
    ax.plot(datos['season'], datos['titulos'],
            marker='o', linewidth=2.5, markersize=8,
            label=grupo, color=colores[grupo])

ax.axvline(x=2022, color='gray', linestyle='--', alpha=0.5)
ax.text(2022.05, 10.5, 'Retiro Federer\n(2022)', fontsize=9, color='gray')
ax.axvline(x=2024, color='gray', linestyle='--', alpha=0.5)
ax.text(2024.05, 10.5, 'Retiro Nadal\n(2024)', fontsize=9, color='gray')

ax.set_title('El fin de la era de los Big 3\nTítulos en Slams y Masters 1000 (2015-2024)',
             fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Temporada', fontsize=12)
ax.set_ylabel('Títulos', fontsize=12)
ax.set_xticks(todos_anos)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('01_big3_vs_nueva_gen.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Visualización 2 — Títulos Individuales

Desglosa los protagonistas de cada bando. Permite ver el declive gradual de Federer, el dominio tardío de Djokovic y la explosión repentina de Sinner y Alcaraz.

In [ ]:
jugadores_interes = big3 + nueva_gen

titulos_individuales = finales[
    finales['winner_name'].isin(jugadores_interes)
].groupby(['season', 'winner_name']).size().reset_index(name='titulos')

colores_big3 = {
    'Roger Federer':  '#1a78cf',
    'Rafael Nadal':   '#e8462a',
    'Novak Djokovic': '#2ca02c'
}

colores_nueva_gen = {
    'Carlos Alcaraz':    '#ff7f0e',
    'Jannik Sinner':     '#1a78cf',
    'Daniil Medvedev':   '#9467bd',
    'Alexander Zverev':  '#e8462a',
    'Stefanos Tsitsipas':'#8c564b',
    'Holger Rune':       '#17becf'
}

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 12))

# Big 3
for jugador, color in colores_big3.items():
    datos = titulos_individuales[titulos_individuales['winner_name'] == jugador]
    datos_completo = pd.DataFrame({'season': todos_anos})
    datos_completo = datos_completo.merge(datos, on='season', how='left')
    datos_completo['titulos'] = datos_completo['titulos'].fillna(0)
    ax1.plot(datos_completo['season'], datos_completo['titulos'],
             marker='o', linewidth=2.5, markersize=8,
             label=jugador, color=color)

ax1.set_title('Big 3 — Títulos individuales en Slams y Masters 1000',
              fontsize=14, fontweight='bold')
ax1.set_ylabel('Títulos', fontsize=11)
ax1.set_xticks(todos_anos)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.axvline(x=2022, color='gray', linestyle='--', alpha=0.5)
ax1.text(2022.05, 5.3, 'Retiro Federer', fontsize=8, color='gray')
ax1.axvline(x=2024, color='gray', linestyle='--', alpha=0.5)
ax1.text(2024.05, 5.3, 'Retiro Nadal', fontsize=8, color='gray')

# Nueva Generación
for jugador, color in colores_nueva_gen.items():
    datos = titulos_individuales[titulos_individuales['winner_name'] == jugador]
    datos_completo = pd.DataFrame({'season': todos_anos})
    datos_completo = datos_completo.merge(datos, on='season', how='left')
    datos_completo['titulos'] = datos_completo['titulos'].fillna(0)
    ax2.plot(datos_completo['season'], datos_completo['titulos'],
             marker='o', linewidth=2.5, markersize=8,
             label=jugador, color=color)

ax2.set_title('Nueva Generación — Títulos individuales en Slams y Masters 1000',
              fontsize=14, fontweight='bold')
ax2.set_xlabel('Temporada', fontsize=11)
ax2.set_ylabel('Títulos', fontsize=11)
ax2.set_xticks(todos_anos)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('02_titulos_individuales.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Visualización 3 — Títulos por Superficie

Revela el secreto del dominio del Big 3: Nadal era imbatible en tierra batida, Djokovic arrasaba en cancha dura, y juntos no dejaban escapatoria en ninguna superficie. Alcaraz emerge como el único de la Nueva Gen que replica esa versatilidad.

In [ ]:
titulos_superficie = finales[
    finales['winner_name'].isin(jugadores_interes)
].groupby(['winner_name', 'surface']).size().reset_index(name='titulos')

orden = titulos_superficie.groupby('winner_name')['titulos'].sum()\
        .sort_values(ascending=False).index

superficies = ['Hard', 'Clay', 'Grass']
colores_superficie = {
    'Hard':  '#4878cf',
    'Clay':  '#e8462a',
    'Grass': '#2ca02c'
}

fig, ax = plt.subplots(figsize=(14, 7))
x = range(len(orden))
ancho = 0.25

for i, superficie in enumerate(superficies):
    valores = []
    for jugador in orden:
        dato = titulos_superficie[
            (titulos_superficie['winner_name'] == jugador) &
            (titulos_superficie['surface'] == superficie)
        ]['titulos'].values
        valores.append(dato[0] if len(dato) > 0 else 0)
    ax.bar([p + i * ancho for p in x], valores,
           width=ancho, label=superficie,
           color=colores_superficie[superficie], alpha=0.85)

ax.set_title('Títulos por jugador y superficie\nSlams y Masters 1000 (2015-2024)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Jugador', fontsize=11)
ax.set_ylabel('Títulos', fontsize=11)
ax.set_xticks([p + ancho for p in x])
ax.set_xticklabels(orden, rotation=30, ha='right', fontsize=10)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('03_titulos_por_superficie.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Visualización 4 — Porcentaje de Dominio por Grupo

El cierre de la historia. Ver el 0% del Big 3 en 2024 frente al ascenso de la Nueva Gen es el dato más contundente del proyecto.

In [ ]:
total_por_ano = titulos_completo.groupby('season')['titulos']\
                .sum().reset_index(name='total')
titulos_pct = titulos_completo.merge(total_por_ano, on='season')
titulos_pct['porcentaje'] = (
    titulos_pct['titulos'] / titulos_pct['total'] * 100
).round(1)

fig, ax = plt.subplots(figsize=(14, 7))

for grupo in todos_grupos:
    datos = titulos_pct[titulos_pct['grupo_ganador'] == grupo]
    ax.fill_between(datos['season'], datos['porcentaje'],
                    alpha=0.3, color=colores[grupo])
    ax.plot(datos['season'], datos['porcentaje'],
            marker='o', linewidth=2.5, markersize=8,
            label=grupo, color=colores[grupo])
    for _, row in datos.iterrows():
        if row['porcentaje'] > 0:
            ax.annotate(f"{row['porcentaje']}%",
                       (row['season'], row['porcentaje']),
                       textcoords="offset points",
                       xytext=(0, 10), ha='center', fontsize=8)

ax.axvline(x=2022, color='gray', linestyle='--', alpha=0.5)
ax.text(2022.05, 95, 'Retiro Federer', fontsize=8, color='gray')
ax.axvline(x=2024, color='gray', linestyle='--', alpha=0.5)
ax.text(2024.05, 95, 'Retiro Nadal', fontsize=8, color='gray')

ax.set_title('Porcentaje de dominio por grupo\nSlams y Masters 1000 (2015-2024)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Temporada', fontsize=11)
ax.set_ylabel('% de títulos', fontsize=11)
ax.set_xticks(todos_anos)
ax.set_ylim(0, 110)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('04_porcentaje_dominio.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Capítulo 1 completo. 4 gráficos guardados.")

---
## 7. Conclusiones

El análisis del dominio del Big 3 entre 2015 y 2024 revela una historia clara de ascenso, consolidación y caída:

- **2015** fue el pico del dominio colectivo: el Big 3 ganó el 77% de los torneos grandes.
- **Djokovic** fue el último en caer y el más dominante en cancha dura, prácticamente sin competencia entre 2018 y 2023.
- **Nadal** construyó su legado casi exclusivamente en tierra batida, donde era prácticamente imbatible.
- **Federer** fue el primero en retirarse (2022), marcando el inicio del fin de la era.
- El cruce histórico ocurrió en **2021-2022**, cuando la Nueva Gen alcanzó al Big 3 en número de títulos.
- **2024** marcó el cierre definitivo: el Big 3 terminó con 0% de títulos grandes por primera vez en una década.
- **Alcaraz** es el único de la Nueva Gen que domina todas las superficies, el heredero más completo del legado del Big 3.
- **Sinner** protagonizó el ascenso más explosivo y repentino, consolidándose como el número 1 en cuanto Djokovic bajó su intensidad.

> *"Durante una década, tres jugadores redefinieron los límites del tenis. En 2024, por primera vez, el tenis les perteneció a otros."*